<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v3/blob/main/notebooks/admin/90_REFRESH_MARKET_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantvesting | Refresh Market Data

Refresh the canonical Screener CSV from the supplied `myScreenerDB.xlsx` source workbook, then create and publish a versioned market-data snapshot.

> **Admin notebook:** this is operational tooling, not part of the customer journey.


In [1]:
!pip install ta -qq

  Preparing metadata (setup.py) ... done


In [2]:
from pathlib import Path
import sys
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT = Path.cwd()
if not (PROJECT / "src" / "quantvesting").exists():
    PROJECT = Path("/content/drive/My Drive/quantvesting_v3")
if not (PROJECT / "src" / "quantvesting").exists():
    raise FileNotFoundError("Open this notebook from the Quantvesting repository or set PROJECT.")
sys.path.insert(0, str(PROJECT / "src"))
from IPython.display import display
from quantvesting import Quantvesting, load_config, load_market_data, load_portfolio_data
config = load_config(PROJECT / "config" / "strategy.yaml")
qv = Quantvesting(config)
MARKET_DATA_DIR = PROJECT / "market_data"
print("Refreshing Screener data from:", MARKET_DATA_DIR / "myScreenerDB.xlsx")
result = qv.ingest_screener(MARKET_DATA_DIR)
print(f"Rows written: {len(result)}")
display(result.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Refreshing Screener data from: /content/drive/My Drive/quantvesting_v3/market_data/myScreenerDB.xlsx
Rows written: 589


,Name,CMP,ATH%,PE,EPS,PB,MCap,ROCE%,ROE%,Sales_Grwth%,Profit_Grwth%,MedPE,ROCE_5Yr%,ROE_5Yr%,Debt2EqR,PAT_12M,CFO_2_EBITDA%,CapType,Symbol,Latest
0,https://www.screener.in/company/RELIANCE/conso...,1419.6,13.54,25.03,61.49,2.19,1921069.29,9.69,8.40,9.01,10.92,26.01,0.34,8.45,0.43,97776.00,98.18,LC,RELIANCE,1
1,https://www.screener.in/company/HDFCBANK/conso...,903.9,12.90,18.67,48.55,2.48,1390929.01,7.51,14.45,5.83,7.09,20.26,0.77,16.13,6.20,77429.81,44.84,LC,HDFCBANK,1
2,https://www.screener.in/company/BHARTIARTL/con...,2004.7,8.48,37.31,53.30,9.70,1143101.05,13.48,23.18,25.05,53.20,62.21,0.62,14.88,1.77,37051.20,106.91,LC,BHARTIARTL,1
3,https://www.screener.in/company/SBIN/consolida...,1198.6,0.43,13.65,91.71,1.87,1106381.82,6.47,17.20,6.22,2.16,10.48,1.26,14.97,10.90,86537.15,11.79,LC,SBIN,1
4,https://www.screener.in/company/ICICIBANK/cons...,1414.6,6.04,19.11,74.19,2.92,1012157.65,7.87,17.89,7.46,7.64,19.38,0.94,16.79,5.51,56609.30,74.67,LC,ICICIBANK,1



## Phase E | Create the canonical market snapshot

The snapshot keeps the existing CSV contracts intact and adds a versioned `metadata.json`. Technical and relative-strength data are materialised through the new `MarketDataProvider` boundary (currently yfinance).

In [3]:
from quantvesting import generate_market_snapshot, R2SnapshotPublisher

SNAPSHOT_DIR = MARKET_DATA_DIR / "snapshots" / "latest"
snapshot = generate_market_snapshot(
    MARKET_DATA_DIR,
    SNAPSHOT_DIR,
    config=config,
    include_derived=True,
    source="yfinance-plus-existing-quantvesting-market-data",
)

print("Snapshot ID:", snapshot.snapshot_id)
print("As of:", snapshot.as_of)
for name, details in snapshot.files.items():
    print(f"{name}: {details['rows']} rows | {details['sha256'][:12]}...")

ERROR:yfinance:$TIPSINDLTD.NS: possibly delisted; no timezone found
ERROR:yfinance:$TV18BRDCST.NS: possibly delisted; no timezone found
ERROR:yfinance:$TIPSINDLTD.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TV18BRDCST.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


Snapshot ID: md_71f43e39518cb4c7
As of: 2026-09-06T10:54:32Z
myScreenerDB.csv: 589 rows | c677fc6764f1...
myProspectsScrips.csv: 292 rows | ac953fb66daa...
myProspects-Momentum.csv: 247 rows | de403b0afbbb...
technical.csv: 290 rows | 93c37472d689...
relative_strength.csv: 290 rows | 2ea576561dad...



### Publish to the same R2 bucket used by the Web UI

This uses the existing Cloudflare Worker admin endpoint, so no R2 S3 credentials are required in Colab.

In [4]:
WORKER_URL = input("Cloudflare Worker URL (blank = skip publish): " ).strip()
if WORKER_URL:
    import getpass
    ADMIN_TOKEN = getpass.getpass("Cloudflare ADMIN_TOKEN: " )
    published = R2SnapshotPublisher(WORKER_URL, ADMIN_TOKEN).publish(SNAPSHOT_DIR, snapshot)
    print("Published:", published['snapshot_id'])
else:
    print("Publish skipped.")

Cloudflare Worker URL (blank = skip publish): https://quantvesting-v3.rathi-ankit.workers.dev/
Cloudflare ADMIN_TOKEN: ··········
Published: md_71f43e39518cb4c7
